# Graph Fraud Detection — Feature Engineering

Registers five Postgres-backed online feature views for graph-based fraud detection:

| Feature View | Type | Entity | Freshness |
|---|---|---|---|
| `GRAPH_COMMUNITY_FEATURES` | Batch | APPLICANT | 1-min DT |
| `ADDRESS_COMMUNITY_FEATURES` | Batch | ADDRESS | 1-min DT |
| `IP_COMMUNITY_FEATURES` | Batch | IP_NETWORK | 1-min DT |
| `STREAM_VELOCITY` | Stream FV | APPLICANT | **2-3 seconds** |
| `COMBINED_FRAUD_FEATURES` | Real-time FV | APPLICANT | Request-time |

Also demonstrates stream freshness and generates a PIT-correct training dataset.

**Prerequisites**: Run `00_setup.ipynb` first (Feature Store + Online Service + entities).

In [1]:
import os
import sys
import time
import numpy
import pandas as pd

import snowflake.connector
from snowflake.snowpark import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import (
    StructType, StructField, StringType, DoubleType,
    TimestampType, TimestampTimeZone,
)
from snowflake.ml.feature_store import (
    FeatureStore,
    FeatureView,
    Entity,
    CreationMode,
    OnlineConfig,
    OnlineStoreType,
    StreamSource,
    StreamConfig,
    RequestSource,
)

# Load PAT from Snowflake config
import tomllib, pathlib
_sf_config = tomllib.load(open(pathlib.Path.home() / '.snowflake' / 'config.toml', 'rb'))
os.environ.setdefault('SNOWFLAKE_PAT', _sf_config['connections']['parker_demo']['password'])
print(f'SNOWFLAKE_PAT loaded ({len(os.environ["SNOWFLAKE_PAT"])} chars)')


SNOWFLAKE_PAT loaded (218 chars)


## Configuration & Connection

In [2]:
CONNECTION_NAME  = 'parker_demo'
DATABASE         = 'GRAPH_FRAUD_DEMO'
FEATURE_SCHEMA   = 'FEATURES'
WAREHOUSE        = 'GRAPH_FRAUD_WH'

In [3]:
from snowflake.ml.feature_store import online_service

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    conn = snowflake.connector.connect(connection_name=CONNECTION_NAME)
    session = Session.builder.configs({'connection': conn}).create()

session.sql_simplifier_enabled = False
session.sql('USE ROLE ACCOUNTADMIN').collect()
session.sql(f'USE DATABASE {DATABASE}').collect()
session.sql(f'USE WAREHOUSE {WAREHOUSE}').collect()

fs = FeatureStore(
    session=session,
    database=DATABASE,
    name=FEATURE_SCHEMA,
    default_warehouse=WAREHOUSE,
    creation_mode=CreationMode.FAIL_IF_NOT_EXIST,
)

applicant_entity = fs.get_entity('APPLICANT')
address_entity  = fs.get_entity('ADDRESS')
ip_entity       = fs.get_entity('IP_NETWORK')

status = fs.get_online_service_status()
print(f'Connected. Online Service: {status.status}')
print(f'  Query  : {online_service.endpoint_url(status, "query")}')
print(f'  Ingest : {online_service.endpoint_url(status, "ingest")}')

You might have more than one threads sharing the Session object trying to update sql_simplifier_enabled. Updating this while other tasks are running can potentially cause unexpected behavior. Please update the session configuration before starting the threads.


Connected. Online Service: RUNNING
  Query  : https://bvciccnd-sfsenorthamerica-perickson-aws1.snowflakecomputing.app
  Ingest : https://nuciccnd-sfsenorthamerica-perickson-aws1.snowflakecomputing.app


## Shared Online Config

In [4]:
online_cfg = OnlineConfig(
    enable=True,
    target_lag='30s',
    store_type=OnlineStoreType.POSTGRES,
)

## FV1: Graph Community Features (Batch, APPLICANT, Postgres Online)

Community stats + edge counts from Leiden results.
Source: `COMMUNITY_RESULTS` + `APPLICANT_EDGES`.

In [5]:
# Graph + address + IP community features, all keyed by APPLICANT_ID
graph_feature_df = session.sql(f"""
    WITH community_stats AS (
        SELECT community_id,
            COUNT(*) AS community_size,
            COUNT(*) > 10 AS is_large_community
        FROM {DATABASE}.GRAPH.COMMUNITY_RESULTS
        WHERE community_id != -1
        GROUP BY community_id
    ),
    edge_counts AS (
        SELECT applicant_id,
            COUNT(*) AS degree,
            COUNT_IF(edge_type = 'SHARED_ADDRESS') AS shared_address_count,
            COUNT_IF(edge_type = 'SHARED_IP')      AS shared_ip_count,
            COUNT_IF(edge_type = 'SHARED_DEVICE')  AS shared_device_count,
            COUNT_IF(edge_type = 'SHARED_BANK')    AS shared_bank_count
        FROM (
            SELECT applicant_a AS applicant_id, edge_type FROM {DATABASE}.EDGES.APPLICANT_EDGES
            UNION ALL
            SELECT applicant_b AS applicant_id, edge_type FROM {DATABASE}.EDGES.APPLICANT_EDGES
        ) GROUP BY applicant_id
    ),
    community_refund_stats AS (
        SELECT cr.community_id,
            AVG(tr.claim_amount) AS avg_community_refund,
            STDDEV(tr.claim_amount) AS community_refund_stddev
        FROM {DATABASE}.GRAPH.COMMUNITY_RESULTS cr
        JOIN {DATABASE}.RAW.APPLICATIONS tr ON tr.applicant_id = cr.applicant_id
        WHERE cr.community_id != -1
        GROUP BY cr.community_id
    ),
    -- Address-level community context (per applicant's filing address)
    applicant_addr AS (
        SELECT applicant_id, address_id
        FROM {DATABASE}.RAW.APPLICATIONS
        QUALIFY ROW_NUMBER() OVER (PARTITION BY applicant_id ORDER BY submitted_at DESC) = 1
    ),
    addr_stats AS (
        SELECT ta.address_id,
            COUNT(DISTINCT cr2.applicant_id) AS num_applicants_at_address,
            MAX(cs2.community_size) AS community_size_at_address,
            AVG(tr2.claim_amount) AS avg_refund_at_address
        FROM applicant_addr ta
        JOIN {DATABASE}.RAW.APPLICATIONS tr2 ON tr2.address_id = ta.address_id
        LEFT JOIN {DATABASE}.GRAPH.COMMUNITY_RESULTS cr2 ON cr2.applicant_id = tr2.applicant_id
        LEFT JOIN community_stats cs2 ON cs2.community_id = cr2.community_id
        GROUP BY ta.address_id
    ),
    -- IP-level community context (per applicant's IP)
    applicant_ip AS (
        SELECT applicant_id, ip_address
        FROM {DATABASE}.RAW.APPLICATIONS
        QUALIFY ROW_NUMBER() OVER (PARTITION BY applicant_id ORDER BY submitted_at DESC) = 1
    ),
    ip_stats AS (
        SELECT ti.ip_address,
            COUNT(DISTINCT cr3.applicant_id) AS num_applicants_at_ip,
            MAX(cs3.community_size) AS community_size_at_ip,
            AVG(tr3.claim_amount) AS avg_refund_at_ip
        FROM applicant_ip ti
        JOIN {DATABASE}.RAW.APPLICATIONS tr3 ON tr3.ip_address = ti.ip_address
        LEFT JOIN {DATABASE}.GRAPH.COMMUNITY_RESULTS cr3 ON cr3.applicant_id = tr3.applicant_id
        LEFT JOIN community_stats cs3 ON cs3.community_id = cr3.community_id
        GROUP BY ti.ip_address
    )
    SELECT
        cr.applicant_id::VARCHAR AS applicant_id,
        cr.community_id,
        COALESCE(cs.community_size, 0) AS community_size,
        COALESCE(cs.is_large_community, FALSE) AS is_large_community,
        COALESCE(ec.degree, 0) AS degree,
        COALESCE(ec.shared_address_count, 0) AS shared_address_count,
        COALESCE(ec.shared_ip_count, 0) AS shared_ip_count,
        COALESCE(ec.shared_device_count, 0) AS shared_device_count,
        COALESCE(ec.shared_bank_count, 0) AS shared_bank_count,
        COALESCE(crs.avg_community_refund, 0) AS avg_community_refund,
        COALESCE(crs.community_refund_stddev, 0) AS community_refund_stddev,
        -- Address context
        COALESCE(ads.num_applicants_at_address, 0) AS addr_n_applicants,
        COALESCE(ads.community_size_at_address, 0) AS addr_community_size,
        COALESCE(ads.avg_refund_at_address, 0) AS addr_avg_refund,
        -- IP context
        COALESCE(ips.num_applicants_at_ip, 0) AS ip_n_applicants,
        COALESCE(ips.community_size_at_ip, 0) AS ip_community_size,
        COALESCE(ips.avg_refund_at_ip, 0) AS ip_avg_refund
    FROM {DATABASE}.GRAPH.COMMUNITY_RESULTS cr
    LEFT JOIN community_stats cs ON cs.community_id = cr.community_id
    LEFT JOIN edge_counts ec ON ec.applicant_id = cr.applicant_id
    LEFT JOIN community_refund_stats crs ON crs.community_id = cr.community_id
    LEFT JOIN applicant_addr ta ON ta.applicant_id = cr.applicant_id
    LEFT JOIN addr_stats ads ON ads.address_id = ta.address_id
    LEFT JOIN applicant_ip ti ON ti.applicant_id = cr.applicant_id
    LEFT JOIN ip_stats ips ON ips.ip_address = ti.ip_address
""")

graph_fv = FeatureView(
    name='GRAPH_COMMUNITY_FEATURES',
    entities=[applicant_entity],
    feature_df=graph_feature_df,
    refresh_freq='5 minutes',
    online_config=online_cfg,
    desc='Graph + address + IP community features, all keyed by APPLICANT_ID.',
)

registered_graph_fv = fs.register_feature_view(graph_fv, 'V1', overwrite=True)
print(f'Registered: {registered_graph_fv.name}/{registered_graph_fv.version}')

for _ in range(20):
    count = fs.read_feature_view(registered_graph_fv, store_type='offline').count()
    if count > 0:
        print(f'  Offline backfill: {count:,} rows')
        break
    time.sleep(15)


POSTGRES online store type is in Public Preview since 1.43.0.
/Users/perickson/miniconda3/envs/customer_work/lib/python3.11/site-packages/snowflake/ml/feature_store/feature_store.py:6002: UserWarning: Your pipeline won't be incrementally refreshed due to: "This dynamic table contains a complex query. Refresh mode has been automatically set to FULL for more predictable performance. To use INCREMENTAL, re-create the dynamic table with REFRESH_MODE=INCREMENTAL. See https://docs.snowflake.com/user-guide/dynamic-table-performance-guide for details.".
  self._check_dynamic_table_refresh_mode(feature_view_name)


Registered: GRAPH_COMMUNITY_FEATURES/V1
  Offline backfill: 2,980,028 rows


## FV2: Address Community Features (Batch, ADDRESS, Postgres Online)

Aggregates community members per filing address so new applicants at known addresses
inherit community context without waiting for Leiden.

In [6]:
address_feature_df = session.sql(f"""
    SELECT
        tr.ADDRESS_ID::VARCHAR AS ADDRESS_ID,
        COUNT(DISTINCT cr.APPLICANT_ID) AS NUM_FILERS_AT_ADDRESS,
        MAX(cs.community_size) AS COMMUNITY_SIZE_AT_ADDRESS,
        AVG(tr.CLAIM_AMOUNT) AS AVG_REFUND_AT_ADDRESS
    FROM {DATABASE}.RAW.APPLICATIONS tr
    LEFT JOIN {DATABASE}.GRAPH.COMMUNITY_RESULTS cr ON cr.APPLICANT_ID = tr.APPLICANT_ID
    LEFT JOIN (
        SELECT community_id, COUNT(*) AS community_size
        FROM {DATABASE}.GRAPH.COMMUNITY_RESULTS
        WHERE community_id != -1
        GROUP BY community_id
    ) cs ON cs.community_id = cr.community_id
    GROUP BY tr.ADDRESS_ID::VARCHAR
""")

address_fv = FeatureView(
    name='ADDRESS_COMMUNITY_FEATURES',
    entities=[address_entity],
    feature_df=address_feature_df,
    refresh_freq='5 minutes',
    online_config=online_cfg,
    desc='Address-level community context for heuristic inheritance',
)
registered_addr_fv = fs.register_feature_view(address_fv, 'V1', overwrite=True)
print(f'Registered: {registered_addr_fv.name}/{registered_addr_fv.version}')

for _ in range(20):
    count = fs.read_feature_view(registered_addr_fv, store_type='offline').count()
    if count > 0:
        print(f'  Offline backfill: {count:,} rows')
        break
    time.sleep(15)

POSTGRES online store type is in Public Preview since 1.43.0.


/Users/perickson/miniconda3/envs/customer_work/lib/python3.11/site-packages/snowflake/ml/feature_store/feature_store.py:6002: UserWarning: Your pipeline won't be incrementally refreshed due to: "This dynamic table contains a complex query. Refresh mode has been automatically set to FULL for more predictable performance. To use INCREMENTAL, re-create the dynamic table with REFRESH_MODE=INCREMENTAL. See https://docs.snowflake.com/user-guide/dynamic-table-performance-guide for details.".
  self._check_dynamic_table_refresh_mode(feature_view_name)


Registered: ADDRESS_COMMUNITY_FEATURES/V1
  Offline backfill: 496,389 rows


## FV3: IP Community Features (Batch, IP_NETWORK, Postgres Online)

Same pattern as address features but keyed on IP address.

In [7]:
ip_feature_df = session.sql(f"""
    SELECT
        tr.IP_ADDRESS,
        COUNT(DISTINCT cr.APPLICANT_ID) AS NUM_FILERS_AT_IP,
        MAX(cs.community_size) AS COMMUNITY_SIZE_AT_IP,
        AVG(tr.CLAIM_AMOUNT) AS AVG_REFUND_AT_IP
    FROM {DATABASE}.RAW.APPLICATIONS tr
    LEFT JOIN {DATABASE}.GRAPH.COMMUNITY_RESULTS cr ON cr.APPLICANT_ID = tr.APPLICANT_ID
    LEFT JOIN (
        SELECT community_id, COUNT(*) AS community_size
        FROM {DATABASE}.GRAPH.COMMUNITY_RESULTS
        WHERE community_id != -1
        GROUP BY community_id
    ) cs ON cs.community_id = cr.community_id
    GROUP BY tr.IP_ADDRESS
""")

ip_fv = FeatureView(
    name='IP_COMMUNITY_FEATURES',
    entities=[ip_entity],
    feature_df=ip_feature_df,
    refresh_freq='5 minutes',
    online_config=online_cfg,
    desc='IP-level community context for heuristic inheritance',
)
registered_ip_fv = fs.register_feature_view(ip_fv, 'V1', overwrite=True)
print(f'Registered: {registered_ip_fv.name}/{registered_ip_fv.version}')

for _ in range(20):
    count = fs.read_feature_view(registered_ip_fv, store_type='offline').count()
    if count > 0:
        print(f'  Offline backfill: {count:,} rows')
        break
    time.sleep(15)

POSTGRES online store type is in Public Preview since 1.43.0.
/Users/perickson/miniconda3/envs/customer_work/lib/python3.11/site-packages/snowflake/ml/feature_store/feature_store.py:6002: UserWarning: Your pipeline won't be incrementally refreshed due to: "This dynamic table contains a complex query. Refresh mode has been automatically set to FULL for more predictable performance. To use INCREMENTAL, re-create the dynamic table with REFRESH_MODE=INCREMENTAL. See https://docs.snowflake.com/user-guide/dynamic-table-performance-guide for details.".
  self._check_dynamic_table_refresh_mode(feature_view_name)


Registered: IP_COMMUNITY_FEATURES/V1
  Offline backfill: 2,696,582 rows


## FV4: Filing Stream Velocity (Stream FV, APPLICANT, 2-3s Freshness)

A stream feature view ingests real-time filing events via the REST Ingest API.
Updated features are queryable in under 3 seconds.

Three components:
1. **StreamSource** — event schema (must match REST payload)
2. **StreamConfig** — Python transformation + historical backfill
3. **FeatureView** with `stream_config` (no `feature_df`)

In [8]:
filing_stream = StreamSource(
    name='APPLICATION_EVENTS',
    schema=StructType([
        StructField('APPLICANT_ID',        StringType()),
        StructField('ADDRESS_ID',  StringType()),
        StructField('IP_ADDRESS',         StringType()),
        StructField('CLAIM_AMOUNT',      DoubleType()),
        StructField('SUBMISSION_HOUR',        DoubleType()),
        StructField('IS_HIGH_CLAIM',     DoubleType()),
        StructField('IS_ODD_HOUR',        DoubleType()),
        StructField('SUBMITTED_AT',          TimestampType(TimestampTimeZone.NTZ)),
    ]),
    desc='Real-time application events for streaming velocity features',
)

fs.register_stream_source(filing_stream)
print(f'Stream source registered: {filing_stream.name}')

StreamSource is in Public Preview since 1.43.0.


Stream source registered: APPLICATION_EVENTS


In [9]:
def process_filing_event(df: pd.DataFrame) -> pd.DataFrame:
    """Enrich raw filing events with velocity signals."""
    df = df[df['APPLICANT_ID'].notna() & (df['APPLICANT_ID'] != '')]
    df['CLAIM_AMOUNT'] = pd.to_numeric(df['CLAIM_AMOUNT'], errors='coerce').fillna(0)
    df['SUBMISSION_HOUR'] = pd.to_numeric(df['SUBMISSION_HOUR'], errors='coerce').fillna(12)
    df['IS_HIGH_CLAIM'] = (df['CLAIM_AMOUNT'] > 10000).astype(float)
    df['IS_ODD_HOUR'] = ((df['SUBMISSION_HOUR'] >= 0) & (df['SUBMISSION_HOUR'] <= 5)).astype(float)
    return df

In [10]:
backfill_df = session.table(f'{DATABASE}.RAW.APPLICATIONS').select(
    F.col('APPLICANT_ID').cast(StringType()).alias('APPLICANT_ID'),
    F.col('ADDRESS_ID').cast(StringType()).alias('ADDRESS_ID'),
    F.col('IP_ADDRESS').cast(StringType()).alias('IP_ADDRESS'),
    F.col('CLAIM_AMOUNT').cast(DoubleType()).alias('CLAIM_AMOUNT'),
    F.hour('SUBMITTED_AT').cast(DoubleType()).alias('SUBMISSION_HOUR'),
    F.when(F.col('CLAIM_AMOUNT') > 10000, F.lit(1.0)).otherwise(F.lit(0.0)).alias('IS_HIGH_CLAIM'),
    F.when(
        (F.hour('SUBMITTED_AT') >= 0) & (F.hour('SUBMITTED_AT') <= 5),
        F.lit(1.0)
    ).otherwise(F.lit(0.0)).alias('IS_ODD_HOUR'),
    F.col('SUBMITTED_AT').cast(TimestampType(TimestampTimeZone.NTZ)).alias('SUBMITTED_AT'),
)

velocity_fv = FeatureView(
    name='STREAM_VELOCITY',
    entities=[applicant_entity],
    timestamp_col='SUBMITTED_AT',
    stream_config=StreamConfig(
        stream_source=filing_stream,
        transformation_fn=process_filing_event,
        backfill_df=backfill_df,
    ),
    online_config=OnlineConfig(
        enable=True,
        target_lag='10s',
        store_type=OnlineStoreType.POSTGRES,
    ),
    desc='Filing velocity: most recent event enriched with risk signals. 2-3s freshness.',
)

registered_velocity_fv = fs.register_feature_view(velocity_fv, 'V1', overwrite=True)
print(f'Registered: {registered_velocity_fv.name}/{registered_velocity_fv.version}')
print(f'Status: {registered_velocity_fv.status}')

POSTGRES online store type is in Public Preview since 1.43.0.


Registered: STREAM_VELOCITY/V1
Status: FeatureViewStatus.STATIC


## Real-Time Feature View (Optional/Experimental)

The `COMBINED_FRAUD_FEATURES` real-time FV assembles all 15 features at query time. 
It works via the Python SDK but is **not used in the production scoring path** because:
- `feature_sources_per_function` only supports one FV per function (documented limitation)
- It fails for new applicants not yet in the graph FV online store

The production scoring path uses parallel OFS REST reads + client-side assembly + direct Gateway call.

## FV5: Combined Fraud Features (Real-time FV, APPLICANT + ADDRESS + IP_NETWORK)

A **real-time feature view** that assembles the full feature vector at request time.
Accepts `CLAIM_AMOUNT` in the request (entity keys are added automatically),
then joins graph, address, IP, and velocity features from the online store.

Output: 15 raw features. The model receives these and produces the fraud prediction.

In [11]:
request_source = RequestSource(
    schema=StructType([
        StructField('CLAIM_AMOUNT', DoubleType()),
    ]),
)

graph_fv_ref    = fs.get_feature_view('GRAPH_COMMUNITY_FEATURES', 'V1')
addr_fv_ref     = fs.get_feature_view('ADDRESS_COMMUNITY_FEATURES', 'V1')
ip_fv_ref       = fs.get_feature_view('IP_COMMUNITY_FEATURES', 'V1')
velocity_fv_ref = fs.get_feature_view('STREAM_VELOCITY', 'V1')

In [12]:
def assemble_features(request_df, graph_df, addr_df, ip_df, velocity_df):
    """Assemble raw features from all sources. No risk score — model handles prediction."""
    import numpy
    community_size = graph_df['COMMUNITY_SIZE'].fillna(0).reset_index(drop=True)
    degree = graph_df['DEGREE'].fillna(0).reset_index(drop=True)
    shared_addr = graph_df['SHARED_ADDRESS_COUNT'].fillna(0).reset_index(drop=True)
    shared_ip = graph_df['SHARED_IP_COUNT'].fillna(0).reset_index(drop=True)
    avg_comm_claim = graph_df['AVG_COMMUNITY_REFUND'].fillna(0).reset_index(drop=True)
    comm_claim_std = graph_df['COMMUNITY_REFUND_STDDEV'].fillna(0).reset_index(drop=True)

    addr_community_size = addr_df['COMMUNITY_SIZE_AT_ADDRESS'].fillna(0).reset_index(drop=True)
    addr_n_applicants = addr_df['NUM_FILERS_AT_ADDRESS'].fillna(0).reset_index(drop=True)
    addr_avg_refund = addr_df['AVG_REFUND_AT_ADDRESS'].fillna(0).reset_index(drop=True)

    ip_community_size = ip_df['COMMUNITY_SIZE_AT_IP'].fillna(0).reset_index(drop=True)
    ip_n_applicants = ip_df['NUM_FILERS_AT_IP'].fillna(0).reset_index(drop=True)
    ip_avg_refund = ip_df['AVG_REFUND_AT_IP'].fillna(0).reset_index(drop=True)

    effective_community_size = numpy.where(
        community_size > 0, community_size,
        numpy.where(addr_community_size > 0, addr_community_size, ip_community_size)
    )

    velocity_submission_hour = velocity_df['SUBMISSION_HOUR'].fillna(12).reset_index(drop=True)
    velocity_is_high = velocity_df['IS_HIGH_CLAIM'].fillna(0).reset_index(drop=True)
    velocity_is_odd_hour = velocity_df['IS_ODD_HOUR'].fillna(0).reset_index(drop=True)

    claim = request_df['CLAIM_AMOUNT'].astype(float).reset_index(drop=True)
    has_direct = (community_size > 0).astype(float)

    return pd.DataFrame({
        'EFFECTIVE_COMMUNITY_SIZE': effective_community_size,
        'DEGREE': degree,
        'SHARED_ADDRESS_COUNT': shared_addr,
        'SHARED_IP_COUNT': shared_ip,
        'AVG_COMMUNITY_REFUND': avg_comm_claim,
        'COMMUNITY_REFUND_STDDEV': comm_claim_std,
        'ADDR_N_FILERS': addr_n_applicants,
        'ADDR_AVG_REFUND': addr_avg_refund,
        'IP_N_FILERS': ip_n_applicants,
        'IP_AVG_REFUND': ip_avg_refund,
        'SUBMISSION_HOUR': velocity_submission_hour,
        'IS_HIGH_CLAIM': velocity_is_high,
        'IS_ODD_HOUR': velocity_is_odd_hour,
        'CLAIM_AMOUNT': claim,
        'HAS_DIRECT_GRAPH': has_direct,
    })

In [13]:
from snowflake.ml.feature_store import RealtimeConfig

combined_fv = FeatureView(
    name='COMBINED_FRAUD_FEATURES',
    entities=[applicant_entity, address_entity, ip_entity],
    realtime_config=RealtimeConfig(
        compute_fn=assemble_features,
        sources=[request_source, graph_fv_ref, addr_fv_ref, ip_fv_ref, velocity_fv_ref],
        output_schema=StructType([
            StructField('EFFECTIVE_COMMUNITY_SIZE', DoubleType()),
            StructField('DEGREE', DoubleType()),
            StructField('SHARED_ADDRESS_COUNT', DoubleType()),
            StructField('SHARED_IP_COUNT', DoubleType()),
            StructField('AVG_COMMUNITY_REFUND', DoubleType()),
            StructField('COMMUNITY_REFUND_STDDEV', DoubleType()),
            StructField('ADDR_N_FILERS', DoubleType()),
            StructField('ADDR_AVG_REFUND', DoubleType()),
            StructField('IP_N_FILERS', DoubleType()),
            StructField('IP_AVG_REFUND', DoubleType()),
            StructField('SUBMISSION_HOUR', DoubleType()),
            StructField('IS_HIGH_CLAIM', DoubleType()),
            StructField('IS_ODD_HOUR', DoubleType()),
            StructField('CLAIM_AMOUNT', DoubleType()),
            StructField('HAS_DIRECT_GRAPH', DoubleType()),
        ]),
    ),
    desc='Real-time assembled feature vector: 15 raw features for XGBoost scoring.',
)

registered_combined_fv = fs.register_feature_view(combined_fv, 'V1', overwrite=True)
print(f'Registered: {registered_combined_fv.name}/{registered_combined_fv.version}')

RealtimeFeatureView is in Public Preview since 1.43.0.


Registered: COMBINED_FRAUD_FEATURES/V1


## Stream Freshness Demo

Ingest a new filing event and measure how quickly it appears in the online store.
Demonstrates **2-3 second end-to-end freshness**.

In [14]:
import os
print(f"PAT set: {'yes' if os.environ.get('SNOWFLAKE_PAT') else 'no'}")

PAT set: yes


In [15]:
from datetime import datetime, timezone

stream_fv_obj = fs.get_feature_view('STREAM_VELOCITY', 'V1')

sample_applicants = session.table(f'{DATABASE}.RAW.APPLICATIONS').select('APPLICANT_ID').distinct().limit(3).collect()
demo_applicant = str(sample_applicants[0]['APPLICANT_ID'])
keys = [[demo_applicant]]

print(f'Demo applicant: {demo_applicant}')
print('Waiting for stream FV online table to be ready...')
before_pd = None
for attempt in range(40):
    try:
        before_pd = fs.read_feature_view(stream_fv_obj, keys=keys, store_type='online')
        print(f'  Online store ready (attempt {attempt + 1})')
        break
    except Exception as e:
        print(f'  [{attempt + 1}] Not ready yet: {str(e)[:80]}')
        time.sleep(15)

if before_pd is None:
    print('ERROR: Online store not ready after 10 min.')
else:
    print('\nBEFORE:')
    print(before_pd.to_string(index=False))

    suspicious_filing = [{
        'APPLICANT_ID':       demo_applicant,
        'ADDRESS_ID': '99999',
        'IP_ADDRESS':        '10.0.0.1',
        'CLAIM_AMOUNT':     float(19999.99),
        'SUBMISSION_HOUR':       float(3),
        'IS_HIGH_CLAIM':    float(1),
        'IS_ODD_HOUR':       float(1),
        'SUBMITTED_AT':         datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S'),
    }]
    print(f'\nIngesting suspicious filing: ${suspicious_filing[0]["CLAIM_AMOUNT"]} at hour 3:00')
    accepted = fs.stream_ingest(stream_source='APPLICATION_EVENTS', records=suspicious_filing)
    print(f'Ingested {accepted} event(s)')

    print('\nPolling online store for freshness...')
    poll_start = time.time()
    freshness_s = None
    after_pd = before_pd
    for attempt in range(30):
        time.sleep(1)
        after_pd = fs.read_feature_view(stream_fv_obj, keys=keys, store_type='online')
        if not after_pd.equals(before_pd):
            freshness_s = time.time() - poll_start
            break

    if freshness_s:
        print(f'\nEnd-to-end freshness: {freshness_s:.1f}s (ingest \u2192 online store)')
        print('\nAFTER:')
        print(after_pd.to_string(index=False))
    else:
        print('Feature unchanged after 30s \u2014 stream may still be backfilling.')

Demo applicant: 2474372
Waiting for stream FV online table to be ready...
  Online store ready (attempt 1)

BEFORE:
APPLICANT_ID ADDRESS_ID IP_ADDRESS  CLAIM_AMOUNT  SUBMISSION_HOUR  IS_HIGH_CLAIM  IS_ODD_HOUR
    2474372              None       None            NaN          NaN             NaN          NaN

Ingesting suspicious filing: $19999.99 at hour 3:00
Ingested 1 event(s)

Polling online store for freshness...

End-to-end freshness: 1.4s (ingest → online store)

AFTER:
APPLICANT_ID ADDRESS_ID IP_ADDRESS  CLAIM_AMOUNT  SUBMISSION_HOUR  IS_HIGH_CLAIM  IS_ODD_HOUR
    2474372             99999   10.0.0.1       19999.99          3.0             1.0          1.0


## Generate Point-in-Time Correct Training Dataset

The training spine is `APPLICATIONS` with `IS_FRAUD` labels from `FRAUD_RING_MEMBERS`.
The Feature Store joins features **as-of** each filing's timestamp.

Stream + real-time FVs are online-only; training uses the batch FVs directly.

In [16]:
spine_df = session.sql(f"""
    SELECT
        tr.APPLICANT_ID,
        tr.RETURN_ID,
        tr.SUBMITTED_AT,
        tr.ADDRESS_ID,
        tr.IP_ADDRESS,
        tr.CLAIM_AMOUNT,
        CASE WHEN frm.APPLICANT_ID IS NOT NULL THEN 1.0 ELSE 0.0 END AS IS_FRAUD
    FROM {DATABASE}.RAW.APPLICATIONS tr
    LEFT JOIN {DATABASE}.RAW.FRAUD_RING_MEMBERS frm
        ON frm.APPLICANT_ID = tr.APPLICANT_ID
""")

print(f'Spine: {spine_df.count():,} rows')
print(f'Fraud rate: {spine_df.filter(F.col("IS_FRAUD") == 1.0).count()} fraud rows')

Spine: 3,000,003 rows
Fraud rate: 2918 fraud rows


In [17]:
training_df = fs.generate_training_set(
    spine_df=spine_df,
    features=[registered_graph_fv],
    spine_timestamp_col='SUBMITTED_AT',
    spine_label_cols=['IS_FRAUD'],
    join_method='cte',
)

print(f'Training set columns: {training_df.columns}')
print(f'Training set rows   : {training_df.count():,}')



Training set columns: ['APPLICANT_ID', 'RETURN_ID', 'SUBMITTED_AT', 'ADDRESS_ID', 'IP_ADDRESS', 'CLAIM_AMOUNT', 'IS_FRAUD', 'COMMUNITY_ID', 'COMMUNITY_SIZE', 'IS_LARGE_COMMUNITY', 'DEGREE', 'SHARED_ADDRESS_COUNT', 'SHARED_IP_COUNT', 'SHARED_DEVICE_COUNT', 'SHARED_BANK_COUNT', 'AVG_COMMUNITY_REFUND', 'COMMUNITY_REFUND_STDDEV', 'ADDR_N_FILERS', 'ADDR_COMMUNITY_SIZE', 'ADDR_AVG_REFUND', 'IP_N_FILERS', 'IP_COMMUNITY_SIZE', 'IP_AVG_REFUND']
Training set rows   : 3,000,003


## Feature Store Summary

In [18]:
print('=== Feature Views ===')
fs.list_feature_views().select('NAME', 'VERSION', 'SCHEDULING_STATE', 'ONLINE_CONFIG').show()

print('\n=== Online Service ===')
status = fs.get_online_service_status()
print(f'Status: {status.status}')
print(f'Endpoints: {[ep.name for ep in status.endpoints]}')

print('\nProceed to 02_train.ipynb')

=== Feature Views ===
--------------------------------------------------------------------------------------------------------------------
|"NAME"                      |"VERSION"  |"SCHEDULING_STATE"  |"ONLINE_CONFIG"                                     |
--------------------------------------------------------------------------------------------------------------------
|ADDRESS_COMMUNITY_FEATURES  |V1         |ACTIVE              |{"enable": true, "target_lag": "30 seconds", "s...  |
|GRAPH_COMMUNITY_FEATURES    |V1         |ACTIVE              |{"enable": true, "target_lag": "30 seconds", "s...  |
|IP_COMMUNITY_FEATURES       |V1         |ACTIVE              |{"enable": true, "target_lag": "30 seconds", "s...  |
|STREAM_VELOCITY      |V1         |NULL                |{"enable": true, "target_lag": "0 seconds", "st...  |
|COMBINED_FRAUD_FEATURES     |V1         |NULL                |{"enable": true, "target_lag": "0 seconds", "st...  |
-------------------------------------------------